In [ ]:
import os
# 禁止 JAX 预占所有显存，这对 A100 并发任务至关重要
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import pennylane as qml
import jax
import jax.numpy as jnp
from jax.experimental import sparse as jsparse  # 引入 JAX 稀疏库
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
import optax
from catalyst import qjit
import time
from math import log2

# 检查 GPU 状态
print(f"JAX Devices: {jax.devices()}")



In [ ]:
# =================== 1. 数据加载与预处理 (保持你的逻辑) ===================
matrix_file = "L=6 N=4.npz"

try:
    H_3 = sp.load_npz(matrix_file)
    print(f"✅ 矩阵加载成功: {H_3.shape}, NNZ: {H_3.nnz}")
except FileNotFoundError:
    # 如果没文件，生成一个随机的用于测试
    print("⚠️ 未找到文件，生成随机矩阵测试...")
    dim = 2**14 # 假设 14 qubits
    H_3 = sp.random(dim, dim, density=0.001, format='csr') + sp.eye(dim)
    H_3 = (H_3 + H_3.T.conj()) / 2

# 辅助函数：Gray Code 排序
def gray_order(n: int):
    return [i ^ (i >> 1) for i in range(2 ** n)]

def get_gray_matrix(H_sparse):
    """
    将稀疏矩阵重排为 Gray Code 顺序。
    注意：为了避免 4GB 稠密矩阵内存峰值，我们直接操作稀疏矩阵索引。
    """
    dim = H_sparse.shape[0]
    n = int(round(log2(dim)))

    # 获取 Gray code 索引
    idx = np.array(gray_order(n))

    # 针对稀疏矩阵的高效切片 (比转为 dense 再切片更省内存)
    # H_gray = P @ H @ P.T
    # 但直接索引更快:
    H_gray = H_sparse[idx, :][:, idx]
    return H_gray, n

print("正在进行 Gray Code 重排...")
H_gray_sparse, n_qubits = get_gray_matrix(H_3)
print(f"重排完成。Qubits: {n_qubits}")


In [ ]:

# =================== 2. 【关键】转换为 JAX 稀疏矩阵 ===================
print("正在转换为 JAX BCOO 格式 (以便 GPU 加速)...")
# BCOO 是 JAX 在 GPU 上最高效的稀疏格式
# 这一步将矩阵数据搬运到 GPU 显存中，且只占用几 MB
H_jax = jsparse.BCOO.from_scipy_sparse(H_gray_sparse)
# 排序索引以获得最佳稀疏乘法性能
H_jax = H_jax.sort_indices()
print("JAX 稀疏矩阵准备就绪。")



In [ ]:
# =================== 3. 电路定义 (编译模式) ===================
depth = 200
steps = 5000

# 使用 lightning.gpu
dev = qml.device("lightning.gpu", wires=n_qubits)

# 【关键优化】使用 @qml.qjit
# 我们不返回 expval，而是返回 state()
# 这样 Catalyst 可以把 200 层循环编译成一个 CUDA Kernel
@qml.qjit
@qml.qnode(dev)
def get_state_compiled(params):
    hf = jnp.zeros(n_qubits, dtype=int)
    qml.BasisState(hf, wires=range(n_qubits))

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d*n_qubits+i], wires=i)

        # 这里的循环会被完全展开并编译，运行时没有 Python 开销
        for j in range(0, n_qubits-1, 2):
            qml.CNOT(wires=[j, (j+1) % n_qubits])
        for j in range(1, n_qubits-1, 2):
            qml.CNOT(wires=[j, (j+1) % n_qubits])

    return qml.state()

# =================== 4. 损失函数 (JAX JIT) ===================
# 将 "电路计算" 和 "稀疏矩阵乘法" 融合在一起
@jax.jit
def loss_fn(params):
    # 1. 极速运行电路，获取 GPU 上的状态向量
    psi = get_state_compiled(params)

    # 2. 在 GPU 上执行稀疏矩阵向量乘法 (Sparse MatVec)
    # 这一步非常快，因为 H_jax 已经在 GPU 上了
    H_psi = H_jax @ psi

    # 3. 计算期望值 <psi | H | psi>
    expectation = jnp.vdot(psi, H_psi)

    return jnp.real(expectation)



In [ ]:
# =================== 5. 训练循环 ===================
n_params = depth * n_qubits
init_params = jnp.zeros(n_params)

# 使用 JAX 的优化器
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(init_params)
params = init_params

print(f"\n🚀 开始在 A100 上训练 | Depth: {depth} | Qubits: {n_qubits}")
print("预编译中 (Warm-up)...")
# 触发一次编译
_ = jax.value_and_grad(loss_fn)(params)
print("编译完成，开始跑！")

start_total = time.time()

for i in range(steps):
    step_start = time.time()

    # 计算梯度和能量
    value, grads = jax.value_and_grad(loss_fn)(params)

    # 更新参数
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    if i % 50 == 0:
        # block_until_ready() 用于等待 GPU 异步计算完成，获取准确时间
        grads.block_until_ready()
        t_sec = time.time() - step_start
        print(f"Step {i:4d} | Energy: {value:.8f} Ha | Time: {t_sec*1000:.2f} ms")

print(f"训练完成，总耗时: {time.time()-start_total:.2f}s")